# Lab — Semantic Search

# Lab — Semantic Search

## Objective

Build a hashing-vector search pipeline over a tiny document set.

## Prerequisites

- Relevant [guided book](../../docs/books/03-language-and-representation/index.md) chapters
- Python 3.10+

## Time estimate

30–45 minutes

## Run

```bash
python main.py
python -m pytest test_lab.py -q
```

## Tasks

1. Inspect token buckets and explain why paraphrases score higher than unrelated docs.
2. Add a hard-negative document that shares tokens but wrong intent.
3. Measure recall@1 on five hand-written queries.
4. List what breaks if you change embedding dimensions.

## Reflection

- What broke first when you changed inputs?
- Which simpler baseline would you compare against in a design review?

## Extensions

- Add another test to `test_lab.py`
- Link your observations to a [concept card](../../docs/concepts/index.md)


In [ ]:
"""Lab 02: an intentionally simple hashing-vector search pipeline."""
from collections import Counter
from hashlib import sha256
from math import sqrt
import re

DOCUMENTS = [
    "Reset a forgotten employee password in the identity portal.",
    "Investigate an unavailable application and service outage.",
    "Submit and approve an expense reimbursement invoice.",
    "Request access to a restricted analytics database.",
]


def tokens(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


def embed(text: str, dimensions: int = 32) -> list[float]:
    counts = Counter(tokens(text))
    vector = [0.0] * dimensions
    for token, count in counts.items():
        bucket = int.from_bytes(sha256(token.encode()).digest()[:4], "big") % dimensions
        vector[bucket] += count
    return vector


def cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na, nb = sqrt(sum(x * x for x in a)), sqrt(sum(y * y for y in b))
    return dot / (na * nb) if na and nb else 0.0


def search(query: str) -> list[tuple[float, str]]:
    q = embed(query)
    return sorted(((cosine(q, embed(doc)), doc) for doc in DOCUMENTS), reverse=True)


if __name__ == "__main__":
    for score, document in search("the application is unavailable"):
        print(f"{score:.3f}  {document}")

## Next steps

- Run `python -m pytest test_lab.py -q` from the lab directory.
- Compare your predictions to actual output.
- See the lab guide on the AIEBOK site for the full catalog.
